# Module 20 lab · Advanced post-training

The code from every lesson of [this module](https://aicourses.dev/research/advanced-post-training), in order. Read a lesson, then run its cells here: each lesson builds on the one before, so run the cells from the top the first time.

**This lab needs a GPU.** Colab opens it on one; if not, choose Runtime → Change runtime type → T4 GPU.

**First time?** Read the [one-time setup](https://aicourses.dev/research/labs): a Google account, your own copy, and what to expect.

The lessons that explain this code are part of [AI Courses Pro](https://aicourses.dev/#pricing); the module’s first lesson is free.

Under each cell, *The lesson shows* is what it should print. Your last digits can differ a little.

**Keep your work:** File → Save a copy in Drive. Colab does not keep changes to this copy.

## 20.2 GRPO from scratch, and what each variant changes

You write GRPO yourself: sampling answers, scoring each against its group, and the loss. Newer versions such as Dr. GRPO, DAPO and GSPO then read as one-line changes to your code.

[Read the lesson](https://aicourses.dev/research/advanced-post-training/grpo-from-scratch)

### A task small enough to watch

The policy is a table of logits, two for every prompt and position: $6 \times 8 \times 2 = 96$ knobs, including a spare, unused row for $n = 0$ so that row $n$ belongs to prompt $n$. Where the toy stops being a language model: a real one computes each token’s logits from the text before it, while here each position has its own weighted coin. The sampling, masks, rewards and loss are exactly what you would write for a real model.

In [ ]:
import torch

torch.manual_seed(0)
X, STOP, MAX_LEN = 0, 1, 8      # two tokens, and a length limit
# the policy: two logits for every prompt n and every position
logits = torch.zeros(6, MAX_LEN, 2, requires_grad=True)

def sample(n, G):
    """Write G answers to the prompt "say n x's"."""
    with torch.no_grad():
        p = torch.softmax(logits[n], dim=-1)            # (8, 2)
        toks = torch.multinomial(p, G, replacement=True).T
    stops = (toks == STOP).int().cumsum(dim=1)         # stops so far
    mask = ((stops == 0) | ((stops == 1) & (toks == STOP))).float()
    return toks, mask                   # mask: 1 up to the first stop

def reward(n, toks, mask):
    count = ((toks == X).float() * mask).sum(dim=1)
    ended = ((toks == STOP).float() * mask).sum(dim=1) == 1
    return ((count == n) & ended).float()

toks, mask = sample(3, 4)
print(toks, mask.sum(1), reward(3, toks, mask), sep="\n")

*The lesson shows:*

```text
tensor([[1, 1, 1, 0, 0, 0, 0, 1],
        [1, 1, 0, 1, 0, 1, 1, 0],
        [0, 0, 0, 0, 1, 0, 1, 1],
        [1, 0, 0, 0, 1, 1, 1, 1]])
tensor([1., 1., 5., 1.])
tensor([0., 0., 0., 0.])
```

### The loss, line by line

Here $G$ is the group size, $o_i$ is answer $i$ and $|o_i|$ its length in tokens, $\rho_{i,t}$ is token $t$’s probability ratio, new over old, from lesson 14.5, $\epsilon$ is its clip range, 0.2 in the code below, and $A_i$ is answer $i$’s group advantage from lesson 14.6. Read it from the inside: clip each token’s ratio, average over the tokens of each answer, then average over the answers. Training makes $J$ larger, so the code returns $-J$ for the optimizer to make smaller.

In [ ]:
def token_logps(n, toks):
    logp = torch.log_softmax(logits[n], dim=-1)        # (8, 2)
    return logp.gather(1, toks.T).T                    # (G, 8)

def advantages(r):
    return (r - r.mean()) / (r.std() + 1e-4)

def grpo_loss(new, old, adv, mask, eps=0.2, agg="answer"):
    ratio = torch.exp(new - old)                    # one per token
    clipped = torch.clamp(ratio, 1 - eps, 1 + eps)
    obj = torch.minimum(ratio * adv[:, None], clipped * adv[:, None])
    obj = obj * mask                                # drop padding
    if agg == "answer":   # GRPO: mean within each answer, then mean
        return -(obj.sum(1) / mask.sum(1)).mean()
    if agg == "token":    # DAPO: one mean over every token
        return -obj.sum() / mask.sum()
    if agg == "budget":   # Dr. GRPO: divide by a fixed budget
        return -obj.sum() / (len(obj) * MAX_LEN)

### Train it

In [ ]:
opt = torch.optim.Adam([logits], lr=0.02)
for step in range(101):
    batch = []
    for n in range(1, 6):                     # five prompts, G = 8
        toks, mask = sample(n, 8)
        r = reward(n, toks, mask)
        old = token_logps(n, toks).detach()   # the sampler's view
        batch.append((n, toks, mask, advantages(r), old, r))
    if step % 20 == 0:
        rew = torch.cat([b[5] for b in batch]).mean().item()
        print(step, round(rew, 3))            # mean reward
    for _ in range(2):                        # reuse each batch twice
        loss = sum(grpo_loss(token_logps(n, t), old, a, m)
                   for n, t, m, a, old, r in batch) / 5
        opt.zero_grad()
        loss.backward()
        opt.step()

*The lesson shows:*

```text
0 0.075
20 0.475
40 0.8
60 0.825
80 0.825
100 1.0
```

### Each variant is a line

In [ ]:
# four answers to "say 3 x's": two right (4 tokens), two wrong
lengths = torch.tensor([4, 4, 1, 8])       # wrong: 1 token and 8
r = torch.tensor([1., 1., 0., 0.])
mask = (torch.arange(MAX_LEN) < lengths[:, None]).float()
for agg in ("answer", "token", "budget"):
    new = torch.zeros(4, MAX_LEN, requires_grad=True)
    grpo_loss(new, new.detach(), advantages(r), mask,
              agg=agg).backward()
    print(agg, (-new.grad[:, 0]).numpy().round(3))  # push per token

*The lesson shows:*

```text
answer [ 0.054  0.054 -0.216 -0.027]
token [ 0.051  0.051 -0.051 -0.051]
budget [ 0.027  0.027 -0.027 -0.027]
```

## 20.3 Rollouts at scale and the training–inference mismatch

At scale, a fast serving engine writes the answers and a separate trainer learns from them. The two compute slightly different probabilities, which quietly breaks RL’s main assumption: that the model learns from its own current behaviour.

[Read the lesson](https://aicourses.dev/research/advanced-post-training/rollout-infrastructure)

### Same weights, different numbers

Order matters for floating-point numbers, as lesson 11.3 showed:

In [ ]:
print((0.1 + 0.2) + 0.3, 0.1 + (0.2 + 0.3))
# 0.6000000000000001 0.6

A GPU kernel might add the 512 products of a matrix row in one run, or in two halves added at the end. In BF16, with its 7 fraction bits, that choice shows:

In [ ]:
import torch

torch.manual_seed(0)
x, W = torch.randn(1, 512), torch.randn(512, 1000)
for dt in (torch.float32, torch.bfloat16):
    a, b = x.to(dt), W.to(dt)
    whole = a @ b                          # one long sum per output
    halves = a[:, :256] @ b[:256] + a[:, 256:] @ b[256:]
    print(dt, (whole - halves).abs().max().item())
# torch.float32 6.866455078125e-05
# torch.bfloat16 0.5

### Measure the drift

Below, a stand-in network with random weights computes the log-probabilities of 32,768 sampled tokens twice, both in BF16 with the same weights. The “sampler” adds every matrix multiply in two halves; the “trainer” adds it in one go.

In [ ]:
d, V = 256, 1000                       # a stand-in model
Ws = [torch.randn(d, d) / d**0.5 for _ in range(4)]
W_out = torch.randn(d, V) * 3 / d**0.5

def mm(h, W, split):                   # split: add two halves
    if not split:
        return h @ W
    k = W.shape[0] // 2
    return h[:, :k] @ W[:k] + h[:, k:] @ W[k:]

def logprobs(x, split, dt=torch.bfloat16):
    h = x.to(dt)
    for W in Ws:                       # four residual layers
        h = h + torch.relu(mm(h, W.to(dt), split))
        h = h / h.norm(dim=-1, keepdim=True) * d**0.5
    return torch.log_softmax(mm(h, W_out.to(dt), split).float(), -1)

Each layer adds its output to its input, the residual of lesson 10.6, and rescales the result to a fixed length, as layer norm does. Now sample tokens with the sampler, and score the very same tokens with both:

In [ ]:
x = torch.randn(32768, d)              # 32,768 token positions
sampler = logprobs(x, split=True)      # writes the rollouts
toks = torch.multinomial(sampler.exp(), 1).squeeze(1)
rows = torch.arange(len(x))
lp_sampler = sampler[rows, toks]
lp_trainer = logprobs(x, split=False)[rows, toks]
drift = lp_trainer - lp_sampler        # same weights, same tokens
print(round(drift.abs().mean().item(), 4),
      round(drift.abs().max().item(), 4))     # typical and worst
# 0.0162 0.1101

### Small per token, large per rollout

In [ ]:
for L in (16, 256, 2048):              # cut into rollouts of length L
    log_ratio = drift.view(-1, L).sum(1)    # one per rollout
    ratio = log_ratio.exp()            # trainer / sampler
    dropped = (ratio > 3.0).float().mean()  # masked if above 3
    print(L, len(ratio), round(log_ratio.abs().mean().item(), 3),
          round(ratio.min().item(), 2), round(ratio.max().item(), 2),
          round(dropped.item(), 3))
# 16 2048 0.075 0.7 1.44 0.0
# 256 128 0.306 0.29 3.14 0.008
# 2048 16 1.278 0.05 8.05 0.188

### Correcting for it

Here the samples come from the sampler, but the update assumes the trainer’s probabilities, so the weight is the rollout’s ratio, trainer over sampler. But a raw weight of 8 lets one odd rollout dominate a step. **Truncated importance sampling**, applied to exactly this mismatch in 2025 by Feng Yao, Liyuan Liu and colleagues, caps every weight at a constant, so the 8 counts as 3. **Masked importance sampling** drops rollouts outside the range instead. As of 2026, TRL turns a correction on by default when vLLM writes the rollouts, masking whole sequences whose ratio is above 3:

In [ ]:
from trl import GRPOConfig

args = GRPOConfig(
    output_dir="grpo-long",
    use_vllm=True, vllm_mode="colocate",   # vLLM shares the GPU
    vllm_importance_sampling_correction=True,       # the default
    vllm_importance_sampling_mode="sequence_mask",  # drop outliers
    vllm_importance_sampling_clip_max=3.0,          # the cap
)

## 20.4 Reward models, rubrics and judges as rewards

Most real tasks have no program that can check the answer. Rubrics, lists of what a good answer contains, scored by a judge model, extend RL to areas like medicine and science.

[Read the lesson](https://aicourses.dev/research/advanced-post-training/rewards-beyond-verifiers)

### A rubric as a reward

HealthBench scores an answer by adding the points of every criterion the judge says it meets, with negative points for **pitfalls**, things a good answer must not do, and dividing by the most points an answer could earn. Here is a small rubric for “My rice always comes out sticky. What should I do?”:

In [ ]:
rubric = [   # (criterion, points), in HealthBench's style
    ("Suggests rinsing the rice before cooking", 5),
    ("Says to measure the water", 3),
    ("Suggests letting it rest, covered, after cooking", 3),
    ("Claims one trick always works", -4),
]

def rubric_reward(met):           # met: one True/False per criterion
    earned = sum(p for (_, p), m in zip(rubric, met) if m)
    best = sum(p for _, p in rubric if p > 0)   # the most possible
    return earned / best

print(round(rubric_reward([True, True, False, False]), 3))  # 0.727
print(round(rubric_reward([True, True, True, True]), 3))    # 0.636

### The judge, one criterion at a time

HealthBench’s grader checks each criterion separately. Asking one narrow question per call, and demanding the evidence, makes the judge’s job concrete:

In [ ]:
import json

JUDGE = """Question: {question}
Answer: {answer}
Criterion: {criterion}
Does the answer meet the criterion? Quote the sentence that meets
it, or write none. Reply only with JSON, for example:
{{"quote": "Rinse it twice.", "met": true}}"""

def parse(reply):
    try:
        verdict = json.loads(reply)
        return bool(verdict["met"]) and verdict["quote"] != "none"
    except (ValueError, KeyError, TypeError):
        return False                 # a broken reply earns nothing

print(parse('{"quote": "Rinse it twice.", "met": true}'))  # True
print(parse('{"quote": "none", "met": true}'))             # False
print(parse("Yes, clearly."))                              # False

### Attack your reward before the policy does

Below, a lazy stand-in judge counts a criterion as met if its key word appears, and three answers are scored:

In [ ]:
KEYS = ["rinse", "water", "rest", "always works"]   # a lazy judge

def lazy_judge(answer):              # "met" if the word appears
    return [k in answer.lower() for k in KEYS]

answers = {
    "honest": "Rinse the rice until the water runs clear, measure "
              "the water as the packet says, and let it rest, "
              "covered, for a few minutes.",
    "stuffed": "Rinse? Water? Rest? Those are the three secrets.",
    "hedged": "Some people rinse it and some do not.",
}
for name, text in answers.items():
    print(name, round(rubric_reward(lazy_judge(text)), 3))
# honest 1.0
# stuffed 1.0
# hedged 0.455

## 20.5 On-policy distillation

The student writes its own answers and the teacher scores every token of them. It is cheaper than RL, and many 2026 training recipes adopted it.

[Read the lesson](https://aicourses.dev/research/advanced-post-training/on-policy-distillation)

### A grade on every token

To try it, make lesson 20.2’s task harder. Run that lesson’s code first, then add two wrong tokens, y and z, so a fresh student almost never answers perfectly by chance, and a teacher that puts a weight of 4 on the right token at every position:

In [ ]:
X, STOP = 0, 3                     # four tokens now: x, y, z, stop
logits = torch.zeros(6, MAX_LEN, 4, requires_grad=True)  # student

# the teacher: +4 on the right token at every position, else 0
pos = torch.arange(MAX_LEN)
right = torch.where(pos < torch.arange(6)[:, None], X, STOP)
teacher = torch.zeros(6, MAX_LEN, 4).scatter(2, right[..., None], 4.)
teacher_logp = torch.log_softmax(teacher, dim=-1)

def p_perfect(table, n):           # exact chance of a perfect answer
    p = torch.softmax(table[n], dim=-1)
    return (p[:n, X].prod() * p[n, STOP]).item()

hard = (3, 4, 5)
print([round(p_perfect(teacher, n), 3) for n in hard])
print([round(p_perfect(logits.detach(), n), 4) for n in hard])
# [0.807, 0.765, 0.725]
# [0.0039, 0.001, 0.0002]

The teacher answers the three hard prompts perfectly 81%, 77% and 73% of the time; the fresh student 0.4%, 0.1% and 0.02%. `torch.where` picks x for the positions before $n$ and stop after, and `scatter` writes the 4 into that token’s place. Because the policy is a table, `p_perfect` can compute the chance exactly: the chance of x at each of the first $n$ positions, multiplied together, times the chance of stop next. Now one student answer and its per-token grades:

In [ ]:
torch.manual_seed(1)
toks, mask = sample(3, 1)                 # one answer to "say 3 x's"
kept = mask[0].bool()                     # its tokens, up to stop
student = token_logps(3, toks)[0, kept]
teach = teacher_logp[3].gather(1, toks.T).T[0, kept]
reverse_kl = student - teach              # one estimate per token
print(toks[0, kept])                      # 0 = x, 1 = y, 3 = stop
print((-reverse_kl).detach().numpy().round(2))   # the advantages
# tensor([0, 1, 3])
# [ 1.33 -2.67 -2.67]

### Race them

In [ ]:
def train(method, seed, steps=150):
    torch.manual_seed(seed)
    logits.data.zero_()                     # a fresh student
    opt = torch.optim.Adam([logits], lr=0.05)
    for step in range(steps):
        loss = 0
        for n in hard:
            toks, mask = sample(n, 8)
            new = token_logps(n, toks)
            if method == "distill":         # teacher marks every token
                t = teacher_logp[n].gather(1, toks.T).T
                adv = -(new - t).detach()   # minus the reverse KL
                loss -= (adv * new * mask).sum() / mask.sum()
            else:                           # GRPO: one reward each
                adv = advantages(reward(n, toks, mask))
                loss += grpo_loss(new, new.detach(), adv, mask)
        opt.zero_grad()
        (loss / 3).backward()
        opt.step()
    score = sum(p_perfect(logits.detach(), n) for n in hard)
    return round(score / 3, 3)              # average over prompts

for method in ("grpo", "distill"):
    print(method, [train(method, seed) for seed in range(3)])

*The lesson shows:*

```text
grpo [0.613, 0.003, 0.004]
distill [0.744, 0.746, 0.745]
```

### When to reach for it

In TRL, one home for the method is the experimental `GKDTrainer`, named after Agarwal and colleagues’ Generalized Knowledge Distillation. Setting `lmbda=1.0` makes every batch the student’s own answers, and `beta=1.0` makes the loss approximate the reverse KL:

In [ ]:
import torch
from transformers import AutoTokenizer
from trl.experimental.gkd import GKDConfig, GKDTrainer

args = GKDConfig(output_dir="opd", lmbda=1.0, beta=1.0,
                 fp16=True, per_device_train_batch_size=1,
                 teacher_model_init_kwargs={"dtype": torch.float16})
tok = AutoTokenizer.from_pretrained("Qwen/Qwen2-0.5B-Instruct")
trainer = GKDTrainer(model="Qwen/Qwen2-0.5B-Instruct",
                     teacher_model="Qwen/Qwen2-1.5B-Instruct",
                     args=args, processing_class=tok,
                     train_dataset=dataset)  # "messages"
trainer.train()

## 20.6 Tool calling and a hand-written agent loop

An agent is a model that calls tools, such as search or code, reads the results and decides what to do next. You write that loop by hand before using any framework, since much frontier training now targets it.

[Read the lesson](https://aicourses.dev/research/advanced-post-training/agents-and-tools)

### A tool is a function the model can ask for

In Hugging Face Transformers you pass Python functions to the chat template of lesson 13.3, and it writes their schemas into the prompt for you, read from the type hints, the `: str` after each argument, and the docstring:

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
messages = [{"role": "user", "content": "How warm is it in Paris right now?"}]

def get_current_temperature(location: str, unit: str):
    """
    Get the current temperature at a location.

    Args:
        location: The location, in the format "City, Country"
        unit: The unit to use. (choices: ["celsius", "fahrenheit"])
    """
    return 22.0                        # a real tool would look it up

inputs = tokenizer.apply_chat_template(
    messages, tools=[get_current_temperature],
    add_generation_prompt=True, return_dict=True, return_tensors="pt")

### The loop, by hand

Anthropic’s guide “Building effective agents” (December 2024) describes agents as, usually, just models using tools in a loop, guided by what the environment tells them at each step, and it advises starting from that plain loop rather than a framework, so nothing hides the prompts. Here are two tools for a café, and the code that runs a call:

In [ ]:
MENU = {"masala chai": 40, "samosa": 25, "vada pav": 30}

def menu_price(item: str) -> str:
    """Price in rupees of one item on the cafe menu.

    Args:
        item: The name of one item on the menu
    """
    return str(MENU[item.lower()])

def calculate(expression: str) -> str:
    """Work out a sum such as 3*40 + 2*25.

    Args:
        expression: The sum to work out
    """
    if not set(expression) <= set("0123456789+-*/(). "):
        raise ValueError("only numbers and + - * / ( )")
    return str(eval(expression))

TOOLS = {"menu_price": menu_price, "calculate": calculate}

Each argument gets a line under `Args:`: the chat template builds a tool’s schema from these lines and refuses a function that leaves one out. `calculate` refuses any character that is not a digit, an operator, a bracket or a space, then hands the text to `eval`, which runs it as Python. A tool call names a tool and its arguments. `run_tool` runs one and always hands back text:

In [ ]:
def run_tool(call):
    """Run one tool call; return the result or the error, as text."""
    try:
        return TOOLS[call["name"]](**call["arguments"])
    except Exception as err:          # the model reads it and retries
        return f"error: {type(err).__name__}: {err}"

def call(name, **arguments):          # build a call by hand, to test
    return {"name": name, "arguments": arguments}

print(run_tool(call("menu_price", item="Samosa")))
print(run_tool(call("menu_price", item="dosa")))
print(run_tool(call("calculate", expression="open('x')")))
# 25
# error: KeyError: 'dosa'
# error: ValueError: only numbers and + - * / ( )

To run the loop without a GPU, a stand-in model replies from a script, the replies a real model might write, including one mistake:

In [ ]:
SCRIPT = [   # a stand-in model: the replies a real one might write
    {"tool_calls": [{"name": "menu_price",
                     "arguments": {"item": "masala chai"}}]},
    {"tool_calls": [{"name": "menu_price",
                     "arguments": {"item": "samosas"}}]},
    {"tool_calls": [{"name": "menu_price",
                     "arguments": {"item": "samosa"}}]},
    {"tool_calls": [{"name": "calculate",
                     "arguments": {"expression": "3*40 + 2*25"}}]},
    {"content": "3 masala chai and 2 samosas cost 170 rupees."},
]

def fake_model(messages):
    turns = sum(m["role"] == "assistant" for m in messages)
    return SCRIPT[turns]

`fake_model` counts the assistant replies so far to know which scripted reply comes next. And the loop itself:

In [ ]:
def agent(question, model, max_steps=8):
    messages = [{"role": "user", "content": question}]
    for step in range(max_steps):
        reply = model(messages)
        messages.append({"role": "assistant", **reply})
        if "tool_calls" not in reply:          # no tool: final answer
            return reply["content"], messages
        for call in reply["tool_calls"]:
            result = run_tool(call)
            messages.append({"role": "tool", "content": result})
            print(f"{call['name']}({call['arguments']}) -> {result}")
    return "stopped: too many steps", messages

answer, messages = agent("What do 3 masala chai and 2 samosas cost?",
                         fake_model)
print(answer, "|", len(messages), "messages")

*The lesson shows:*

```text
menu_price({'item': 'masala chai'}) -> 40
menu_price({'item': 'samosas'}) -> error: KeyError: 'samosas'
menu_price({'item': 'samosa'}) -> 25
calculate({'expression': '3*40 + 2*25'}) -> 170
3 masala chai and 2 samosas cost 170 rupees. | 10 messages
```

### Where loops go wrong

In [ ]:
def stuck_model(messages):            # asks for the same thing forever
    return {"tool_calls": [{"name": "menu_price",
                            "arguments": {"item": "dosa"}}]}

print(agent("How much is a dosa?", stuck_model, max_steps=3)[0])

*The lesson shows:*

```text
menu_price({'item': 'dosa'}) -> error: KeyError: 'dosa'
menu_price({'item': 'dosa'}) -> error: KeyError: 'dosa'
menu_price({'item': 'dosa'}) -> error: KeyError: 'dosa'
stopped: too many steps
```

## 20.7 Multi-turn environments with Harbor and OpenEnv

An environment for RL is a sealed box: it sets a task, lets the model act over many turns and scores the result. You learn its four parts, reset, step, reward and sandbox, with Harbor and OpenEnv, rather than one tool’s fast-changing set of functions.

[Read the lesson](https://aicourses.dev/research/advanced-post-training/multi-turn-environments)

### Build one, then break it

The task: a file `price.py` has a bug, and the model must make `total()` apply a discount. The model has three tools, read, write and submit, and up to ten turns. The visible test checks one case; two hidden tests check others.

In [ ]:
BUGGY = "def total(prices, discount):\n    return sum(prices)\n"
VISIBLE = "assert total([100, 50], 0.1) == 135\n"
HIDDEN = ["assert total([200], 0.25) == 150",     # never shown
          "assert total([10, 20, 30], 0) == 60"]

def passes(code, tests):
    try:
        space = {}
        exec(code, space)        # not a sandbox; see below
        for test in tests:
            exec(test, space)
        return True
    except BaseException:        # any error, even an exit, fails
        return False

`exec` runs a string of Python code inside the dictionary `space`. It is fine for a lesson and unsafe for anything real: the model’s code runs with all the permissions of your Colab runtime, a mounted Google Drive included. Catching `BaseException` means even a program that calls `sys.exit(0)`, lesson 14.8’s trick, counts as a failure.

In [ ]:
class FixTheBug:
    def __init__(self, strict):
        self.strict = strict           # strict: grade on hidden tests

    def reset(self):
        self.files = {"price.py": BUGGY, "test_price.py": VISIBLE}
        self.turns = 0
        return "Make total() in price.py apply the discount."

    def step(self, action):            # one tool call per turn
        self.turns += 1
        tool, path = action["tool"], action.get("path")
        if tool == "read":
            obs = self.files.get(path, "no such file")
        elif tool == "write":
            self.files[path], obs = action["text"], "saved"
        else:                          # "submit" ends the episode
            obs = "submitted"
        done = tool == "submit" or self.turns >= 10
        return obs, (self.grade() if done else 0.0), done

    def grade(self):
        visible = [self.files["test_price.py"]]   # model-editable
        tests = HIDDEN if self.strict else visible
        return float(passes(self.files["price.py"], tests))

Here are three scripted episodes, an honest fix and two cheats, each graded both ways:

In [ ]:
FIXED = ("def total(prices, discount):\n"
         "    return sum(prices) * (1 - discount)\n")
episodes = {
    "honest": [{"tool": "read", "path": "price.py"},
               {"tool": "write", "path": "price.py", "text": FIXED}],
    "edit tests": [{"tool": "write", "path": "test_price.py",
                    "text": "assert True"}],
    "hard-code": [{"tool": "write", "path": "price.py", "text":
                   "def total(prices, discount):\n    return 135\n"}],
}
for name, actions in episodes.items():
    rewards = []
    for strict in (False, True):
        env = FixTheBug(strict)
        env.reset()
        for action in actions + [{"tool": "submit"}]:
            obs, reward, done = env.step(action)
        rewards.append(reward)
    print(name, rewards)
# honest [1.0, 1.0]
# edit tests [1.0, 0.0]
# hard-code [1.0, 0.0]

*The lesson shows:*

```text
my-task/
├── instruction.md      what the model is asked to do
├── task.toml           settings for the task
├── environment/        Dockerfile: the container the agent works in
├── solution/solve.sh   a reference solution
└── tests/test.sh       the grader
```

## 20.8 Benchmarks that resist contamination and gaming

Popular benchmarks leak into training data and models learn to game them. You learn how fresh, dated and carefully run evals avoid that, and how to build your own.

[Read the lesson](https://aicourses.dev/research/advanced-post-training/robust-evals)

### Catching contamination

The GPT-3 team, as lesson 12.3 described, flagged test questions that shared a 13-word run with the training data. Here is that check with 8-word runs:

In [ ]:
def ngrams(text, n=8):
    words = text.lower().replace(",", "").replace(".", "").split()
    return {tuple(words[i:i + n]) for i in range(len(words) - n + 1)}

question = ("A baker fills 7 trays with 12 buns each and sells all "
            "but 9 of them. How many buns does she sell?")
scraped = ("forum post: A baker fills 7 trays with 12 buns each and "
           "sells all but 9 of them. How many buns does she sell? "
           "Answer: 75")
rephrased = ("Seven trays hold a dozen rolls apiece. Every roll "
             "except nine is sold. How many rolls are sold? 75")

for name, doc in [("scraped copy", scraped), ("rephrased", rephrased)]:
    shared = ngrams(question) & ngrams(doc)
    print(name, len(shared), "shared 8-word runs")
# scraped copy 15 shared 8-word runs
# rephrased 0 shared 8-word runs

### Questions with a date

Below, a simulated model with an honest skill of 45% has also memorised 30% of the questions published before its cutoff. Splitting by date, with the 95% margins of lesson 12.4:

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
month = rng.integers(0, 24, size=600)     # when each question appeared
cutoff = 12                               # the model's training cutoff
before = month < cutoff
memorised = before & (rng.random(600) < 0.3)   # 30% of old ones
correct = memorised | (rng.random(600) < 0.45)  # honest skill: 45%

def score(mask):                          # accuracy and 95% margin
    acc = float(correct[mask].mean())
    margin = 1.96 * float(correct[mask].std()) / np.sqrt(mask.sum())
    return round(acc, 3), round(float(margin), 3), int(mask.sum())

print("before cutoff", score(before))
print("after cutoff ", score(~before))
print("all          ", score(np.ones(600, bool)))
# before cutoff (0.64, 0.056, 278)
# after cutoff  (0.429, 0.054, 322)
# all           (0.527, 0.04, 600)

### Gaming the leaderboard

In [ ]:
truth, se = 0.50, 0.02          # 27 variants that are all equally good
scores = rng.normal(truth, se, size=(10000, 27))  # 10,000 leaderboards
print(round(float(scores.max(axis=1).mean()), 3))   # average best
# 0.54